# The golden routing dataset

One row per labelled `(dataset, query_id)`: the three per-route objective
scores, the derived route, and the outcome shape. A label comes from running
`dense_only` / `pure_rrf` / `sparse_only` against the lane's indexed corpus
and scoring each ranking with `0.7·HitRate@1 + 0.3·NDCG@10` — never from
asking a model which route looks right. The build pipeline lives in
`notebooks/route_labels.ipynb`; this notebook only reads the artifact.

In [13]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
import pandas as pd

from composition import CellFill
from hybrid_search_rrf_dataset.labels import RouteLabels
from hybrid_search_rrf_dataset.objective import RouterObjective

selection = CellFill().build(force=True)
labels = RouteLabels(selection, objective=RouterObjective(min_relevance=1))
data = labels.load()
print(f"{len(data):,} labelled rows | {data['dataset'].nunique()} lanes | "
      f"{data['query_id'].nunique():,} distinct queries")
print(f"objective: {labels.objective.name}")

52,331 labelled rows | 42 lanes | 49,846 distinct queries
objective: 0.7*HitRate@1+0.3*NDCG@10


In [15]:

pd.set_option('display.max_colwidth', None)

selection.sample(n=10)[['query', 'cell']]

,query,cell
6199,Can NPP-VIIRS data be used to calibrate and generate synthetic OLS radiance data for the period after 2013?,number_inside_natural_question
5304,Any reason NOT to run Linux in a VM all the time?,acronym_inside_question
687,but i do song,short_grammatical_question
4502,I can't figure out how Google Earth Engine supports the generation of high-resolution burned area maps.,negation_bearing_question
2751,"I will use the programming language pony.\nProblem:\nAn array is monotonic if it is either monotone increasing or monotone decreasing. An array nums is monotone increasing if for all i <= j, nums[i] <= nums[j]. An array nums is monotone decreasing if for all i <= j, nums[i] >= nums[j]. Given an integer array nums, write a function that returns true if the given array is monotonic, or false otherwise.\n\nHere is the code template:\nfun isMonotonic(nums: Array[I32]): Bool =>\n...",extreme_length_pasted_query
7443,What procedures are used to collect and analyze zircon U-Pb ages and Hf isotopic data in the context of geochemical studies of granitic rocks?,
642,what is state id,short_grammatical_question
2537,"Angular use injection outside constructor, directly in class attribute\nMy student is asking me : << why should I inject stuff inside the constructor instead of injecting directly in the attribute of the class ?\nWhat I teach to her :\nUse injection inside the constructor\nhousingLocationList: HousingLocation[] = [];\nhousingService: HousingService = inject(HousingService);\n\nconstructor() {\n this.housingLocationList = this.housingService.getAllHousingLocations();\n}\n\nWhat She wants to do :\nInject the housing service directly inside the class attribute\n@Component({\n//...\n})\nexport class HomeComponent {\n\n housingService: HousingService = inject(HousingService);\n housingLocationList: HousingLocation[] = this.housingService.getAllHousingLocations();\n \n constructor() {}\n}\n\nWhat should I answer to her ?\nWhat I tried :\nI tried to convice her that it's a dogma and she should not think about it and just do it like that :)\nWhat I expected :\nShe accept my answer\nWhat actually resulted:\nShe still wants to know",extreme_length_pasted_query
2688,"Finish the following code based on the docstring: fn main(){}\n\nuse std::{slice::Iter, cmp::{max, self}, mem::replace, collections::{HashSet, HashMap}, ops::Index, ascii::AsciiExt};\nuse rand::Rng;\nuse regex::Regex;\nuse md5;\nuse std::any::{Any, TypeId};\n\n/*\n From a supplied list of numbers (of length at least two) select and return two that are the closest to each\n other and return them in order (smaller number, larger number).\n \n*/\nfn find_closest_elements(numbers:Vec<f32>) -> (f32,f32){\n\n",extreme_length_pasted_query
7459,which foods contain kefir,


## 1 — What's in it

Shape decides what a row can teach. `routes_differ` carries the
dense-vs-sparse signal, `all_tied` says any route works, `all_zero` found
nothing relevant and stores no label.

In [7]:
shapes = data["shape"].value_counts().rename_axis("shape").to_frame("rows")
shapes["share"] = (shapes["rows"] / len(data) * 100).round(1)
shapes

,rows,share
shape,,
routes_differ,23363,49.9
all_tied,15162,32.4
all_zero,8331,17.8


### What a tie looks like

A row ties for one of two reasons, and the score it ties AT tells them apart:
ties at **1.0** mean every route put a judged answer at rank 1 — the corpus is
too easy for routing to matter. Ties at **lower equal scores** mean the routes
retrieved different top-10s whose differing documents were never judged —
unjudged docs score zero, so the measured scores coincide (the d60 finding:
zero of ~15K ties are irreducible; the differing tails sit unjudged in the
cached rankings). The first kind is fixed by harder corpora, the second by
judging the tails.

In [19]:
import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

In [29]:
from pathlib import Path

from hybrid_search_rrf_dataset.lanes import LANES

DATA = Path("data")

tied = data[data["shape"] == "all_tied"]

tie_share = (
    data.groupby("dataset")["shape"]
    .apply(lambda s: (s == "all_tied").mean())
    .sort_values(ascending=False)
)
print("tie-heaviest lanes (share of the lane's rows):")
display(tie_share.head(8).round(3).rename("all_tied share"))

print("what score the routes tie AT (1.0 = judged doc at rank 1 for all three):")
display(
    tied["score_dense_only"].round(2).value_counts(normalize=True)
    .head(6).round(3).rename("share of ties")
)


def gold_answer(dataset: str, query_id: str, chars: int = 240) -> str:
    """First judged doc's text head — read with a parquet doc_id filter so
    big corpora never fully load."""
    source = LANES[dataset].source.name if dataset in LANES else dataset
    min_rel = LANES[dataset].min_relevance if dataset in LANES else 1
    qrels = pd.read_parquet(DATA / source / "qrels.parquet")
    ids = qrels.loc[
        (qrels["query_id"] == query_id) & (qrels["relevance"] >= min_rel),
        "doc_id",
    ].head(1).tolist()
    if not ids:
        return "(no judged doc at min_relevance)"
    docs = pd.read_parquet(
        DATA / source / "corpus.parquet", filters=[("doc_id", "in", ids)]
    )
    return docs["text"].iloc[0][:chars] if len(docs) else "(gold doc missing from corpus)"


examples = tied.groupby("dataset").head(1).sort_values("dataset").head(12).copy()
examples["gold_answer"] = [
    gold_answer(row.dataset, row.query_id) for row in examples.itertuples()
]
examples[["dataset", "query", "gold_answer",
          "score_dense_only", "score_pure_rrf", "score_sparse_only"]]

tie-heaviest lanes (share of the lane's rows):


dataset
miracl-en-dev              0.654
webfaq-eng                 0.601
msmarco-passage-dev        0.491
gooaq                      0.453
rarb-math                  0.378
orcas                      0.285
clerc                      0.280
lotte-technology-search    0.271
Name: all_tied share, dtype: float64

what score the routes tie AT (1.0 = judged doc at rank 1 for all three):


score_dense_only
1.00    0.956
0.19    0.024
0.15    0.005
0.88    0.004
0.13    0.002
0.93    0.001
Name: share of ties, dtype: float64

,dataset,query,gold_answer,score_dense_only,score_pure_rrf,score_sparse_only
48552,beir-nfcorpus,aneurysm,OBJECTIVES: To test the efficacy and safety of oral L-citrulline supplementation in improving erection hardness in patients with mild erectile dysfunction (ED). L-arginine supplementation improves nitric oxide-mediated vasodilation and endo,0.807686,0.807686,0.807686
8341,bright-biology,"Do animals exhibit handedness (paw-ness?) preference?\nI have been observing my cat and found that when confronted with an unknown item, she will always use her front left paw to touch it.\nThis has me wondering if animals exhibit handedness like humans do? (and do I have a left handed cat?)\nOne note of importance is that with an unknown item, her approach is always identical, so possibly using the left paw means allowing a fast possible exit based on how she positions her body.\nThis question is related to Are there dextral/sinistral higher animals?. However, I question the ""paw-ness"" as a consequence of how the cat is approaching new items (to be ready to flee), whereas the other question remarks about the high number of ""right-pawed"" dogs and questions the influence of people for this preference.",In other animals[edit]\nIt has been shown that cerebral lateralization is a widespread phenomenon in the animal kingdom. Functional and structural differences between left and right brain hemispheres can be found in many other vertebrates an,1.000000,1.000000,1.000000
8722,bright-earth-science,I have never understood why earth's inner core is solid. Considering that the inner core is made of an iron-nickel alloy (melting point around 1350 C to 1600 C) and the temperature of the inner core is approximately 5430 C (about the temperature of the surface of the sun). Since Earth's core is nearly 3-4 times the melting point of iron-nickel alloys how can it possibly be solid?,"part1 -------------------\n\nEarth's inner core is the innermost geologic layer of the planet Earth. It is primarily a solid ball with a radius of about 1,220 km (760 mi), which is about 20% of Earth radius or 70% of the Moon's radius.\nThere",0.883944,0.883944,0.883944
8558,bright-economics,"Quadratic Form Single Summation notation\n\nI'm trying to translate what feels like a double summation notation , compacted into a single summation. It's the summation definition of quadratic forms.\n\n𝑄(𝑥1,𝑥2.....𝑥𝑛)=∑𝑖≤𝑗𝑎𝑖𝑗𝑥𝑖𝑥𝑗\n\nI assume quadratic forms can be written as the less compact double summation?\n\n𝑄(𝑥1,𝑥2.....𝑥𝑛)=∑𝑛𝑖=1∑𝑛𝑗=1𝑎𝑖𝑗𝑥𝑖𝑥𝑗\n\nTo count this we would:\n\nFix the outer summation first, so in this case we would be fixing rows and going across columns. Alternately we could switch the summation so we fixed columns and counted rows. This holds for any 𝑛•𝑚\n matrix i believe?\nQuestion: I suppose the 𝑖≤𝑗\n notation has to correspond to this. But it's not immediately clicking?","3 Double Summation\nSometimes data are grouped in a table (or matrix) like the one below:\n""\n1 2 3\n4 5 6\n#\n.\n3\nThis table has 2 rows and 3 columns and is therefore called a 2×3 matrix (to be\nread as “a 2-by-3 matrix”). Suppose that the number",1.000000,1.000000,1.000000
48715,bright-leetcode,"Given an array of integers `citations` where `citations[i]` is the number of citations a researcher received for their `ith` paper, return _the researcher's h-index_.\n\nAccording to the [definition of h-index on Wikipedia](https://en.wikipedia.org/wiki/H-index): The h-index is defined as the maximum value of `h` such that the given researcher has published at least `h` papers that have each been cited at least `h` times.\n\n**Example 1:**\n\n**Input:** citations = \[3,0,6,1,5\]\n**Output:** 3\n**Explanation:** \[3,0,6,1,5\] means the researcher has 5 papers in total and each of them had received 3, 0, 6, 1, 5 citations respectively.\nSince the researcher has 3 papers with at least 3 citations each and the remaining two with no more than 3 citations each, their h-index is 3.\n\n**Example 2:**\n\n**In

In [27]:
# verify one tie by hand: what does the query share with its judged answer?
import re

from wordfreq import zipf_frequency

from hybrid_search_rrf_dataset.retrieval import SnapshotDataset

PROBE_TEXT = "Do animals exhibit handedness"  # change to inspect another tie

row = tied[tied["query"].str.startswith(PROBE_TEXT)].iloc[0]
lane = row["dataset"]
source_name = LANES[lane].source.name if lane in LANES else lane
source = SnapshotDataset(source_name, path="data")
min_rel = LANES[lane].min_relevance if lane in LANES else 1

qrels = source.qrels()
gold_ids = qrels.loc[
    (qrels["query_id"] == row["query_id"]) & (qrels["relevance"] >= min_rel),
    "doc_id",
].tolist()
corpus = source.corpus().set_index("doc_id")["text"]

tokens = lambda t: set(re.findall(r"[a-z0-9]+", t.lower()))  # noqa: E731
q_tokens = tokens(row["query"])
print(f"lane: {lane} | gold docs: {len(gold_ids)} | "
      f"scores d/r/s: {row['score_dense_only']:.2f}/"
      f"{row['score_pure_rrf']:.2f}/{row['score_sparse_only']:.2f}")

for doc_id in gold_ids[:2]:
    d_tokens = tokens(str(corpus.get(doc_id, "")))
    shared = q_tokens & d_tokens
    rare = sorted(shared, key=lambda w: zipf_frequency(w, "en"))[:8]
    print(f"\ngold {doc_id}: query↔gold overlap "
          f"{len(shared) / max(len(q_tokens), 1):.0%} of query terms")
    print(f"  rarest shared terms (Zipf asc): {rare}")
    print(f"  doc head: {str(corpus.get(doc_id, ''))[:180]}")

lane: bright-biology | gold docs: 2 | scores d/r/s: 1.00/1.00/1.00

gold animals_handedness/Laterality_2.txt: query↔gold overlap 52% of query terms
  rarest shared terms (Zipf asc): ['handedness', 'paw', 'preference', 'exhibit', 'approaching', 'exit', 'whereas', 'handed']
  doc head: In other animals[edit]
It has been shown that cerebral lateralization is a widespread phenomenon in the animal kingdom. Functional and structural differences between left and right

gold animals_handedness/Handedness_7.txt: query↔gold overlap 23% of query terms
  rarest shared terms (Zipf asc): ['handedness', 'preference', 'handed', 'dogs', 'animals', 'however', 'left', 'use']
  doc head: In other animals[edit]
Kangaroos and other macropod marsupials show a left-hand preference for everyday tasks in the wild. 'True' handedness is unexpected in marsupials however, be


In [28]:
# the trap, quantified: how much of the dataset is vocabulary-coincident?
gold = pd.read_parquet("data/encoder_router/gold_doc_profile.parquet")
merged = data.merge(
    gold[["dataset", "query_id", "gold.overlap"]],
    on=["dataset", "query_id"], how="left",
)
answerable = merged[merged["shape"] != "all_zero"].copy()

print("query↔gold overlap by outcome shape:")
display(answerable.groupby("shape")["gold.overlap"].describe()[["mean", "50%"]].round(3))

answerable["overlap_band"] = pd.cut(
    answerable["gold.overlap"], [0, 0.2, 0.4, 0.6, 0.8, 1.0], include_lowest=True
)
bands = answerable.groupby("overlap_band", observed=True).agg(
    rows=("query_id", "size"),
    sparse_perfect=("score_sparse_only", lambda s: (s >= 0.99).mean()),
    dense_perfect=("score_dense_only", lambda s: (s >= 0.99).mean()),
    tied=("shape", lambda s: (s == "all_tied").mean()),
).round(3)
print("\nby overlap band — if sparse_perfect climbs with overlap, the")
print("benchmark-vocabulary trap is real, and the LOW bands are the")
print("production-realistic slice:")
bands

query↔gold overlap by outcome shape:


,mean,50%
shape,,
all_tied,0.662,0.667
routes_differ,0.595,0.577



by overlap band — if sparse_perfect climbs with overlap, the
benchmark-vocabulary trap is real, and the LOW bands are the
production-realistic slice:


,rows,sparse_perfect,dense_perfect,tied
overlap_band,,,,
"(-0.001, 0.2]",1785,0.151,0.529,0.125
"(0.2, 0.4]",7169,0.382,0.498,0.263
"(0.4, 0.6]",11473,0.519,0.584,0.414
"(0.6, 0.8]",10719,0.562,0.607,0.476
"(0.8, 1.0]",9980,0.469,0.523,0.404


In [8]:
differ = data[data["shape"] == "routes_differ"]
routes = differ["route"].value_counts().rename_axis("route").to_frame("wins")
routes["share of trainable"] = (routes["wins"] / len(differ) * 100).round(1)
routes

,wins,share of trainable
route,,
dense_only,14056,60.2
sparse_only,7917,33.9
pure_rrf,1390,5.9


The corpora are why routing has something to learn: lanes differ in average
IDF and out-of-vocabulary share, and the best blind route moves with them.

In [9]:
stats = pd.read_parquet("data/route_labels/lane_corpus_stats.parquet")
print(stats.groupby("target")["avg_idf"].agg(["count", "mean"]).round(3).to_string())
pd.concat([stats.nlargest(3, "avg_idf"), stats.nsmallest(3, "avg_idf")])[
    ["dataset", "avg_idf", "oov_share", "target"]
].round(3)

             count   mean
target                   
dense_only      13  0.391
sparse_only      2  0.511


,dataset,avg_idf,oov_share,target
9,crumb-theorem-retrieval,0.572,0.143,sparse_only
4,crumb-code-retrieval,0.538,0.038,dense_only
0,beir-nfcorpus,0.484,0.017,dense_only
6,crumb-paper-retrieval,0.280,0.000,dense_only
5,crumb-legal-qa,0.293,0.000,dense_only
7,crumb-set-operation-entity-retrieval,0.318,0.000,dense_only


## 2 — One winner per query: the decisive core

The argmax form keeps a row for training only when one route clearly won.
Decisive means the winner put a relevant document at rank 1 and the
runner-up missed — under this objective exactly margin ≥ 0.4, derived from
the weights, not hand-picked. The bar is deliberately strict, and it keeps
about one row in nine. Twice the pre-repair count (the IDF and
selection-join fixes), but the other eight rows stay invisible to training.

In [10]:
from hybrid_search_rrf_dataset.router import decisive_rows

before = pd.read_parquet(labels.labels_path.parent / "labels.backup-pre-idf-full.parquet")
now, then = decisive_rows(data), decisive_rows(before)
print(f"decisive now:        {len(now):,} of {len(data):,} rows")
print(f"decisive before fix: {len(then):,} of {len(before):,} rows")
now["winner"].value_counts().rename_axis("winner").to_frame("decisive wins")

decisive now:        5,183 of 46,856 rows
decisive before fix: 2,607 of 20,815 rows


,decisive wins
winner,
dense_only,3203
sparse_only,1753
pure_rrf,227


## 3 — Three yes/no labels: ties become signal

SPEC d60 changes the label form, not the scores. Each route gets
`ok = score ≥ oracle − 0.3`, where 0.3 is the objective's own `ndcg_weight`
— the widest gap two routes can show while still sharing the same top-1
outcome. A tied row labels `[1, 1, 1]` and trains all three classifiers;
`all_zero` rows stay null. At tolerance 0 the view reproduces the stored
`route` column exactly, so nothing is overwritten — the same scores read a
second way.

In [11]:
view = labels.acceptability().frame()
answerable = view[view["serve"].notna()]
print(f"decisive rows:      {len(now):,}")
print(f"acceptability rows: {len(answerable):,}  "
      f"({len(answerable) / len(now):.1f}x, zero new retrieval)")

pd.DataFrame({
    route: answerable[f"ok_{route}"].astype(bool).value_counts()
    for route in ("dense_only", "pure_rrf", "sparse_only")
}).T.rename(columns={True: "acceptable", False: "not acceptable"})

decisive rows:      5,183
acceptability rows: 38,525  (7.4x, zero new retrieval)


,acceptable,not acceptable
dense_only,33853,4672
pure_rrf,33565,4960
sparse_only,29925,8600


`serve` — the cheapest acceptable route — is the evaluation target, not a
training label. On tied rows it points at sparse, which is where the class
that argmax starved gets its training diet.

In [12]:
serve = answerable["serve"].value_counts().rename_axis("serve").to_frame("rows")
serve["share"] = (serve["rows"] / len(answerable) * 100).round(1)
serve

,rows,share
serve,,
sparse_only,29925,77.7
dense_only,8373,21.7
pure_rrf,227,0.6


## 4 — Growing the thin archetypes: augmentation

Half the archetype cells cannot fill their draw from natural supply
(`notebooks/selection_audit.ipynb` §3); those cells are generation targets. An
operator edits a real parent query under a declared meaning-preserving
transformation, so the child inherits the parent's human judgments (d43d).
`data/augmentation/pool.parquet` holds the accepted children.

In [58]:
from composition.cells import CELLS_BY_NAME
from composition.compose import join_text
from dataset_registry import DATASETS

pool = pd.read_parquet("data/augmentation/pool.parquet")
for_cells = pool[pool["floor"].isin(CELLS_BY_NAME)]
print(f"{len(pool):,} accepted children | {len(for_cells):,} minted for "
      f"{for_cells['floor'].nunique()} archetype cells, the rest for "
      f"pre-cell band floors")

sample = for_cells.groupby("floor").head(1).head(6).copy()
as_parents = sample.drop(columns=["query_id"]).rename(
    columns={"parent_dataset": "dataset", "generated_from": "query_id"})
sample["parent"] = join_text(as_parents, {d.name: d for d in DATASETS}).values
sample.rename(columns={"floor": "cell"})[["cell", "operator", "parent"]].assign(
    parent=sample["parent"].str.slice(0, 70),
    child=sample["query"].str.slice(0, 70),
)

2,366 accepted children | 377 minted for 22 archetype cells, the rest for pre-cell band floors


,cell,operator,parent,child
1989,legal_citation_canonical,inject,Are eviction cases first heard in circuit court? In the state of North,North Carolina eviction proceedings 42 U.S.C. district court jurisdict
2019,datetime_token_present,inject,[Image],events on 2017-03-31 23:26:44
2031,single_token_char_blob,stat_rewrite,Why do people drink Cuervo when it tastes the same going down as comin,Cuervo_identical_taste_ingestion_regurgitation_equivalencexx
2039,business_temporal_reference,inject,2021 films about action and comedy.,action and comedy films released during Q4 2021
2044,symbol_pile_no_grammar,inject,alternative medicine,cPGES lipoxinA4 alternative medicine
2074,bibliographic_catalog_identifier,inject,Problem: Brook Hills High School currently enrolls 3000 students. Half,Brook Hills High School 3000-1125 female students enrollment


### One round, priced before it is paid for

`campaign.plan()` shows the whole pass — one row per hungry floor with the
operator it would use and the row count it targets — without an LLM call.
The commented line underneath runs one real round for a single floor and
appends its accepted children to the pool.

In [19]:
from augmentation.campaign import AugmentationCampaign
from augmentation.config import AugmentationPaths
from augmentation.loop import AugmentationLoop
from augmentation.parents import ParentPool
from dataset_registry import DATASETS

paths = AugmentationPaths()
catalog = pd.read_parquet(paths.catalog).astype({"query_id": str})
parents = ParentPool(catalog, selection.astype({"query_id": str}),
                     {d.name: d for d in DATASETS})
loop = AugmentationLoop(selection, sheet_path=paths.cell_order_sheet, parents=parents)

plan = AugmentationCampaign(loop).plan()
plan.head(10)

parents: dropped 3 row(s) with no query text
parents: dropped 8 row(s) with no query text
parents: dropped 2 row(s) with no query text
parents: dropped 1 row(s) with no query text
parents: dropped 6 row(s) with no query text
parents: dropped 5 row(s) with no query text
parents: dropped 2 row(s) with no query text
campaign plan — 22 hungry floors, 284 target rows (>= 284 LLM calls):
                           floor  missing                operator              gate             action  target_rows
        legal_citation_canonical    400.0    inject, stat_rewrite    coherence_gate skip: pilot staged            0
          datetime_token_present    399.0    inject, stat_rewrite    coherence_gate            produce           13
          single_token_char_blob    398.0            stat_rewrite declaration_audit            produce           22
     business_temporal_reference    398.0            stat_rewrite declaration_audit            produce           25
          symbol_pile_no_grammar   

,floor,missing,operator,gate,action,target_rows
0,legal_citation_canonical,400.0,"inject, stat_rewrite",coherence_gate,skip: pilot staged,0
1,datetime_token_present,399.0,"inject, stat_rewrite",coherence_gate,produce,13
2,single_token_char_blob,398.0,stat_rewrite,declaration_audit,produce,22
3,business_temporal_reference,398.0,stat_rewrite,declaration_audit,produce,25
4,symbol_pile_no_grammar,398.0,"inject, stat_rewrite",coherence_gate,skip: pilot staged,0
5,bibliographic_catalog_identifier,398.0,"inject, stat_rewrite",coherence_gate,skip: pilot staged,0
6,bio_clinical_identifier,398.0,"inject, stat_rewrite",coherence_gate,skip: pilot staged,0
7,travel_transport_code,397.0,"inject, stat_rewrite",coherence_gate,produce,25
8,standards_compliance_lookup,394.0,"inject, stat_rewrite",coherence_gate,produce,9
9,boolean_operator_query,393.0,operator_syntax_rewrite,declaration_audit,produce,28


In [20]:
# one real round for the first plannable floor — spends LLM tokens:
floor = plan[plan["action"] == "produce"].iloc[0]["floor"]
loop.run(floor, n=1)

parents: dropped 3 row(s) with no query text
NOTE: 'inject' is gated by coherence_gate (d42h) — rows are produced as feature-stock (no floor credit; skipped by the mini-fill) until the gate's pilot passes.
NOTE: 'stat_rewrite' is gated by declaration_audit (d42h) — rows are produced as feature-stock (no floor credit; skipped by the mini-fill) until the gate's pilot passes.


augment:datetime_token_present:   0%|          | 0/1 [00:01<?, ?row/s, attempted=1, dropped=1]

- 2046371: dropped — failed [length_words >= 3.0 and <= 9.999999999 (measured 12.0)]
    before: 'functions of three regions of sm intestine'
    tried:  'functions of three sm intestine regions 2014-09-17 16:59:58'


augment:datetime_token_present:   0%|          | 0/1 [00:12<?, ?row/s, attempted=2, dropped=2]

- 372: dropped — rounds_exhausted — rounds called: ['generate_surface', 'list_features', 'verify', 'verify', 'generate_surface', 'verify']
    before: 'You are given two circles. Find the area of their intersection.'
    tried:  (nothing — never got that far)


...

augment:datetime_token_present:   0%|          | 0/1 [01:57<?, ?row/s, attempted=11, dropped=11]

- 4301: dropped — rounds_exhausted — rounds called: ['list_features', 'verify', 'verify', 'verify', 'verify', 'verify']
    before: 'You are given a sequence of length N: A_1, A_2, ..., A_N.\nFor each integer i bet'
    tried:  (nothing — never got that far)


augment:datetime_token_present: 100%|██████████| 1/1 [02:04<00:00, 124.79s/row, attempted=12, dropped=11]

+ 610 (1 attempt(s))
    before: 'Cop’s son needs blood transfusion from an inmate .\n A cop’s son is very sick and'
    after:  'cop son blood transfusion inmate 1998-01-30'
datetime_token_present: accepted 1/12 attempts (need 1, parents available 161, minting ['inject', 'stat_rewrite']) -> /Users/andrei/projects/hybrid-search-rrf-dataset/src/data/augmentation/pool.parquet
  spend: 65 hops, 447,720 tokens, 125.3s wall (123.8s of it LLM) -> 65.0 hops/row, 447,720 tokens/row, 125.3s/row


,query_id,query,floor,operator,provenance,generated_from,parent_dataset,home_lane,grounding_doc_id,meaning_preserved,answer_key,attempts,hops,tokens,elapsed_s,credit_gate
0,aug-datetime_token_present-610,cop son blood transfusion inmate 1998-01-30,datetime_token_present,inject,doc_grounded,610,crumb-tip-of-the-tongue,crumb-tip-of-the-tongue,2093587:1,False,minted,1,3,20258,6.922873,coherence_gate


### Already produced, against the current ask

`pool.parquet` accumulates every accepted child across all past campaign
runs; `plan`'s `target_rows` is a snapshot of what the order sheet still
asks for as of the last composition build. A gate-free floor's produced rows
only leave `target_rows` once `CellFill().admit()` folds them into the
composition, so a floor can show rows already produced here while
`target_rows` hasn't moved yet — the two numbers answer different questions,
not the same one twice.

In [28]:
pool = pd.read_parquet("data/augmentation/pool.parquet")  # fresh read — the in-memory `pool` above predates the last run
produced = pool["floor"].value_counts()

progress = plan[plan["target_rows"] > 0].set_index("floor")[
    ["operator", "gate", "target_rows"]
].copy()
progress["produced"] = produced.reindex(progress.index, fill_value=0).astype(int)
progress["remaining"] = (progress["target_rows"] - progress["produced"]).clip(lower=0)
progress.sort_values("produced", ascending=False)

,operator,gate,target_rows,produced,remaining
floor,,,,,
env_var_configuration,inject,coherence_gate,5,25,0
standards_compliance_lookup,"inject, stat_rewrite",coherence_gate,9,21,0
geo_coordinate_postal,"inject, stat_rewrite",coherence_gate,11,19,0
datetime_token_present,"inject, stat_rewrite",coherence_gate,13,18,0
logistics_catalog_token,"inject, stat_rewrite",coherence_gate,15,15,0
code_symbol_named_in_prose,"inject, stat_rewrite",coherence_gate,16,14,2
bare_acronym,stat_rewrite,declaration_audit,17,13,4
single_token_char_blob,stat_rewrite,declaration_audit,22,8,14
status_code_idf_split,"inject, stat_rewrite",coherence_gate,22,8,14


Before/after, word-level, one example per floor: the full parent text, then
the full child text with the diff marked inline — additions in green,
removed words struck through in red. Nothing tracks *why* this particular
parent was picked over another eligible one: `demand()` hands `run()` the
first catalog row a cell's dispatch plan judges servable, in catalog order —
positional, not scored, and the choice is never written to the row.

In [46]:
import difflib

from IPython.display import HTML, display

from scripts.collection_features import CollectionIndexStore

_store = CollectionIndexStore()
_built_lanes = {lane for lane in _store.indexable().values() if _store.path(lane).exists()}


def _pick_example(rows: pd.DataFrame) -> pd.Series:
    """First produced row whose home_lane already has a built corpus index —
    so the corpus-stats cell below can reuse this exact example instead of a
    second, unrelated one."""
    indexed = rows[rows["home_lane"].isin(_built_lanes)]
    return (indexed if not indexed.empty else rows).iloc[0]


def _diff_html(before: str, after: str) -> str:
    """Word-level diff: kept words plain, additions green, removals red
    strikethrough. The full `after` text is still all here — this is not a
    changes-only view, every word survives, some just carry markup."""
    b, a = before.split(), after.split()
    spans = []
    for tag, i1, i2, j1, j2 in difflib.SequenceMatcher(None, b, a).get_opcodes():
        if tag == "equal":
            spans.append(" ".join(a[j1:j2]))
        if tag in ("delete", "replace"):
            spans.append(
                f'<span style="color:#b00020;text-decoration:line-through">'
                f'{" ".join(b[i1:i2])}</span>'
            )
        if tag in ("insert", "replace"):
            spans.append(
                f'<span style="color:#0a7d1f;font-weight:600">'
                f'{" ".join(a[j1:j2])}</span>'
            )
    return " ".join(spans)


sample = pd.DataFrame([
    _pick_example(rows)
    for _, rows in pool[pool["floor"].isin(progress.index)].groupby("floor")
])
as_parents = sample.drop(columns=["query_id"]).rename(
    columns={"parent_dataset": "dataset", "generated_from": "query_id"})
sample["parent"] = join_text(as_parents, {d.name: d for d in DATASETS}).values

order = progress.sort_values("produced", ascending=False).index
rows = sample.set_index("floor").reindex(order)

blocks = [
    f'<div style="margin-bottom:16px;line-height:1.5">'
    f'<b>{floor}</b> <span style="color:#888">'
    f'({row["operator"]}, lane: {row["home_lane"]})</span><br>'
    f'before: {row["parent"]}<br>'
    f'after:&nbsp; {_diff_html(row["parent"], row["query"])}'
    f"</div>"
    for floor, row in rows.iterrows()
]
display(HTML("".join(blocks)))

### Same corpus, measured before and after

The six Query-Corpus scalars (`avg_idf`, `max_idf`, `oov_share`,
`collection_size`, `avg_doc_length`, `vocab_overlap` — SPEC d47b) computed
against each example's own `home_lane`, for the parent and the child text
alike, using the retriever's own BM25 tokenizer. Only lanes with a built
`corpus_index.parquet` show numbers (`src/scripts/collection_features.py`
builds the rest); a floor whose example lane isn't indexed yet reports
`None` rather than a guess.

These are descriptive, not a verdict — a route still needs the router (or a
real retrieval run) to say who wins for a given query. Read `avg_idf`/`max_idf`
rising and `oov_share` rising as evidence pointing toward sparse (more for
exact match to grip, less for BM25 to score at all); the reverse points
toward dense. Nothing here computes an actual route prediction.

In [47]:
from query_taxonomy.corpus_relative import CORPUS_RELATIVE_BANKS


def _corpus_stats(text: str, index) -> dict[str, float]:
    tokens = _store.tokenizer.tokens(text)
    banks = [cls(index) for cls in CORPUS_RELATIVE_BANKS]
    return {stat.name: stat.value for bank in banks for stat in bank.compute(tokens)}


records = []
for floor, row in rows.iterrows():
    lane = row["home_lane"]
    if lane not in _built_lanes:
        records.append({
            "floor": floor, "home_lane": lane,
            "stat": None, "before": None, "after": None, "delta": None,
        })
        continue
    index = _store.load(lane)
    before, after = _corpus_stats(row["parent"], index), _corpus_stats(row["query"], index)
    records.extend(
        {
            "floor": floor, "home_lane": lane, "stat": stat,
            "before": before[stat], "after": after[stat],
            "delta": after[stat] - before[stat],
        }
        for stat in before
    )
corpus_stats = pd.DataFrame(records).round(3)
corpus_stats

,floor,home_lane,stat,before,after,delta
0,env_var_configuration,bright-aops,avg_idf,0.460,0.463,0.003
1,env_var_configuration,bright-aops,max_idf,0.925,1.000,0.075
2,env_var_configuration,bright-aops,oov_share,0.000,0.034,0.034
3,env_var_configuration,bright-aops,collection_size,4.000,4.000,0.000
4,env_var_configuration,bright-aops,avg_doc_length,2.035,2.035,0.000
...,...,...,...,...,...,...
80,rare_key_buried_in_chatter,crumb-legal-qa,max_idf,0.491,0.774,0.284
81,rare_key_buried_in_chatter,crumb-legal-qa,oov_share,0.000,0.000,0.000
82,rare_key_buried_in_chatter,crumb-legal-qa,collection_size,4.000,4.000,0.000
83,rare_key_buried_in_chatter,crumb-legal-qa,avg_doc_length,2.461,2.461,0.000


In [48]:
corpus_stats[corpus_stats["floor"] == "single_token_char_blob"]

,floor,home_lane,stat,before,after,delta
42,single_token_char_blob,bright-aops,avg_idf,0.314,0.328,0.014
43,single_token_char_blob,bright-aops,max_idf,0.627,0.334,-0.293
44,single_token_char_blob,bright-aops,oov_share,0.000,0.000,0.000
45,single_token_char_blob,bright-aops,collection_size,4.000,4.000,0.000
46,single_token_char_blob,bright-aops,avg_doc_length,2.035,2.035,0.000
47,single_token_char_blob,bright-aops,vocab_overlap,0.221,0.049,-0.172
